In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.1 MB/s eta 0:00:00


# Augumentation on Original dataset

In [ ]:
# ============================================================
# INSTALL LIBRARIES
# ============================================================
!pip install albumentations opencv-python -q

import os
import cv2
import random
import albumentations as A
import numpy as np

# ============================================================
# DATASET PATHS
# ============================================================

image_dir = "/content/drive/MyDrive/mes_project/MES_DATASET"
label_dir = "/content/drive/MyDrive/mes_project/kitti_label"

out_image_dir = "/content/drive/MyDrive/mes_project/MES_DATASET_aug"
out_label_dir = "/content/drive/MyDrive/mes_project/kitti_label_aug"

os.makedirs(out_image_dir, exist_ok=True)
os.makedirs(out_label_dir, exist_ok=True)

# ============================================================
# SCENE + POINT LEVEL AUGMENTATIONS
# ============================================================

transform = A.Compose(
[
    # ---------- SCENE LEVEL ----------
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.4),

    # safer replacement for ShiftScaleRotate
    A.Affine(
        scale=(0.9,1.1),
        translate_percent=(-0.05,0.05),
        rotate=(-10,10),
        p=0.5
    ),

    # ---------- POINT LEVEL ----------
    A.RandomBrightnessContrast(p=0.4),
    A.GaussNoise(p=0.3),
    A.MotionBlur(p=0.2),
    A.RandomShadow(p=0.3),

],
bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"])
)

# ============================================================
# OBJECT LEVEL AUGMENTATION (SAFE COPY PASTE)
# ============================================================

def copy_paste_object(image, bbox):

    h, w, _ = image.shape
    x, y, bw, bh = bbox

    # convert YOLO bbox → pixel coords
    x1 = int((x - bw/2) * w)
    y1 = int((y - bh/2) * h)
    x2 = int((x + bw/2) * w)
    y2 = int((y + bh/2) * h)

    x1 = max(0,x1)
    y1 = max(0,y1)
    x2 = min(w,x2)
    y2 = min(h,y2)

    obj = image[y1:y2, x1:x2]

    if obj.size == 0:
        return image

    obj_h, obj_w = obj.shape[:2]

    # choose safe paste position
    nx = random.randint(0, max(0, w - obj_w))
    ny = random.randint(0, max(0, h - obj_h))

    # clip region safely
    paste_region = image[ny:ny+obj_h, nx:nx+obj_w]

    min_h = min(paste_region.shape[0], obj_h)
    min_w = min(paste_region.shape[1], obj_w)

    image[ny:ny+min_h, nx:nx+min_w] = obj[0:min_h, 0:min_w]

    return image

# ============================================================
# AUGMENT DATASET
# ============================================================

augmentations_per_image = 3

for img_name in os.listdir(image_dir):

    if not img_name.endswith((".jpg",".png",".jpeg")):
        continue

    img_path = os.path.join(image_dir, img_name)
    label_path = os.path.join(label_dir, img_name.split(".")[0] + ".txt")

    image = cv2.imread(img_path)

    bboxes = []
    labels = []

    with open(label_path) as f:
        for line in f.readlines():
            cls, x, y, w, h = map(float, line.split())
            bboxes.append([x,y,w,h])
            labels.append(int(cls))

    for i in range(augmentations_per_image):

        aug = transform(image=image, bboxes=bboxes, class_labels=labels)

        aug_img = aug["image"]
        aug_boxes = aug["bboxes"]
        aug_labels = aug["class_labels"]

        # ---------- OBJECT LEVEL ----------
        if len(aug_boxes) > 0 and random.random() < 0.5:

            idx = random.randint(0, len(aug_boxes)-1)
            aug_img = copy_paste_object(aug_img, aug_boxes[idx])

        new_img = f"aug_{i}_{img_name}"
        new_lbl = f"aug_{i}_{img_name.split('.')[0]}.txt"

        cv2.imwrite(os.path.join(out_image_dir,new_img), aug_img)

        with open(os.path.join(out_label_dir,new_lbl),"w") as f:
            for cls,box in zip(aug_labels,aug_boxes):
                x,y,w,h = box
                f.write(f"{cls} {x} {y} {w} {h}\n")

print("Augmentation completed successfully.")

In [ ]:
# ============================================================
# ANALYZE AUGMENTED DATASET (ROBUST VERSION)
# ============================================================

import os
from collections import Counter
import matplotlib.pyplot as plt

label_dir = "/content/drive/MyDrive/MES PROJECT GROUP 6/MES PROJECT Dataset/augmented_labels"

classes = {
0:"car",
1:"motorcycle",
2:"bicycle",
3:"pedestrian",
4:"auto",
5:"bus",
6:"truck",
7:"tractor"
}

object_counts = Counter()
image_counts = Counter()
only_class_images = Counter()
combination_counts = Counter()

total_images = 0
multi_class_images = 0
total_objects = 0
empty_labels = 0

for file in os.listdir(label_dir):

    if not file.endswith(".txt"):
        continue

    total_images += 1
    path = os.path.join(label_dir,file)

    classes_in_image = []

    with open(path) as f:
        lines = f.readlines()

    if len(lines) == 0:
        empty_labels += 1
        continue

    for line in lines:

        parts = line.split()

        if len(parts) < 5:
            continue

        cls = int(float(parts[0]))   # <-- FIX HERE

        object_counts[cls] += 1
        classes_in_image.append(cls)
        total_objects += 1

    unique_classes = set(classes_in_image)

    for c in unique_classes:
        image_counts[c] += 1

    if len(unique_classes) == 1:
        only_class_images[list(unique_classes)[0]] += 1

    if len(unique_classes) > 1:
        multi_class_images += 1

    combination_counts[tuple(sorted(unique_classes))] += 1


# ============================================================
# PRINT STATISTICS
# ============================================================

print("\nTOTAL AUGMENTED IMAGES:", total_images)
print("EMPTY LABEL FILES:", empty_labels)

print("\nTOTAL OBJECTS:", total_objects)

print("\nOBJECT COUNTS")
for k,v in object_counts.items():
    print(classes[k],":",v)

print("\nIMAGE COUNTS")
for k,v in image_counts.items():
    print(classes[k],":",v)

print("\nONLY-CLASS IMAGES")
for k,v in only_class_images.items():
    print(classes[k],":",v)

print("\nMULTI-CLASS IMAGES:", multi_class_images)

print("\nAVERAGE OBJECTS PER IMAGE")
print(total_objects/total_images)


# ============================================================
# CLASS COMBINATIONS
# ============================================================

print("\nTOP CLASS COMBINATIONS")

for combo,count in combination_counts.most_common(10):

    names=[classes[c] for c in combo]
    print(names,":",count)

# ============================================================
# OBJECT DISTRIBUTION GRAPH
# ============================================================

names=[classes[c] for c in object_counts.keys()]
counts=list(object_counts.values())

plt.figure()
plt.bar(names,counts)
plt.xticks(rotation=45)
plt.title("Augmented Dataset Object Distribution")
plt.show()

KeyboardInterrupt: 

In [ ]:
import os
import cv2
from ultralytics import YOLO
from collections import Counter
import matplotlib.pyplot as plt

# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO("yolov8x.pt")

# ============================================================
# DATASET PATHS
# ============================================================

orig_images = "/content/drive/MyDrive/mes_project/MES_DATASET"
orig_labels = "/content/drive/MyDrive/mes_project/original_labels"

aug_images = "/content/drive/MyDrive/mes_project/MES_DATASET_aug"
aug_labels = "/content/drive/MyDrive/mes_project/augmented_labels"

os.makedirs(orig_labels, exist_ok=True)
os.makedirs(aug_labels, exist_ok=True)

# ============================================================
# CLASS DEFINITIONS
# ============================================================

classes = {
0:"car",
1:"motorcycle",
2:"bicycle",
3:"pedestrian",
4:"auto",
5:"bus",
6:"truck",
7:"tractor",
8:"animal",
9:"unknown"
}

# COCO -> dataset mapping
vehicle_map = {
0:3,
1:2,
2:0,
3:1,
5:5,
7:6
}

animal_classes = [15,16,17,18,19,20,21,22,23,24]

# ============================================================
# LABEL GENERATION FUNCTION
# ============================================================

def generate_labels(image_folder, label_folder):

    for img_name in os.listdir(image_folder):

        if not img_name.endswith((".jpg",".png",".jpeg")):
            continue

        img_path = os.path.join(image_folder,img_name)
        img = cv2.imread(img_path)

        results = model(img)[0]

        label_path = os.path.join(label_folder,img_name.split(".")[0]+".txt")

        with open(label_path,"w") as f:

            for box in results.boxes:

                coco_cls = int(box.cls[0])
                conf = float(box.conf[0])

                if conf < 0.35:
                    continue

                x,y,w,h = box.xywhn[0]

                if coco_cls in vehicle_map:

                    cls = vehicle_map[coco_cls]

                    aspect = float(w)/float(h)

                    if cls == 0 and aspect < 0.8:
                        cls = 4

                    if cls == 6 and aspect < 0.9:
                        cls = 7

                elif coco_cls in animal_classes:

                    cls = 8

                else:
                    continue

                f.write(f"{cls} {x} {y} {w} {h}\n")

# ============================================================
# DATASET ANALYSIS FUNCTION
# ============================================================

def analyze_dataset(label_dir, title):

    object_counts = Counter()
    image_counts = Counter()
    combination_counts = Counter()

    total_images = 0
    total_objects = 0
    multi_class_images = 0

    for file in os.listdir(label_dir):

        if not file.endswith(".txt"):
            continue

        total_images += 1

        path = os.path.join(label_dir,file)

        classes_in_image = []

        with open(path) as f:
            lines = f.readlines()

        for line in lines:

            parts = line.split()

            if len(parts) < 5:
                continue

            cls = int(float(parts[0]))

            object_counts[cls]+=1
            classes_in_image.append(cls)
            total_objects += 1

        unique = set(classes_in_image)

        for c in unique:
            image_counts[c]+=1

        if len(unique) > 1:
            multi_class_images += 1

        combination_counts[tuple(sorted(unique))]+=1

    print("\n==============================")
    print("DATASET ANALYSIS:",title)
    print("==============================")

    print("\nTOTAL IMAGES:",total_images)
    print("TOTAL OBJECTS:",total_objects)

    print("\nOBJECT COUNTS")
    for k,v in object_counts.items():
        print(classes[k],":",v)

    print("\nIMAGE COUNTS")
    for k,v in image_counts.items():
        print(classes[k],":",v)

    print("\nMULTI CLASS IMAGES:",multi_class_images)

    print("\nAVERAGE OBJECTS PER IMAGE")
    if total_images > 0:
        print(total_objects/total_images)

    print("\nTOP CLASS COMBINATIONS")

    for combo,count in combination_counts.most_common(10):

        names=[classes[c] for c in combo]
        print(names,":",count)

    # distribution graph
    names=[classes[c] for c in object_counts.keys()]
    counts=list(object_counts.values())

    plt.figure()
    plt.bar(names,counts)
    plt.xticks(rotation=45)
    plt.title(title + " Object Distribution")
    plt.show()


# ============================================================
# RUN LABEL GENERATION
# ============================================================

print("Generating labels for ORIGINAL dataset...")
generate_labels(orig_images, orig_labels)

print("Generating labels for AUGMENTED dataset...")
generate_labels(aug_images, aug_labels)

# ============================================================
# RUN ANALYSIS
# ============================================================

analyze_dataset(orig_labels,"Original Dataset")

analyze_dataset(aug_labels,"Augmented Dataset")

print("\nPipeline completed successfully.")

# Fog Addition on Dataset

In [ ]:
import cv2
import numpy as np
import torch
import os
import glob

def guided_filter(guide, src, radius, eps):
    guide = guide.astype(np.float32) / 255.0
    src = src.astype(np.float32)

    ksize = (2 * radius + 1, 2 * radius + 1)
    mean_I = cv2.boxFilter(guide, -1, ksize)
    mean_p = cv2.boxFilter(src, -1, ksize)
    mean_Ip = cv2.boxFilter(guide * src, -1, ksize)
    cov_Ip = mean_Ip - mean_I * mean_p

    mean_II = cv2.boxFilter(guide * guide, -1, ksize)
    var_I = mean_II - mean_I * mean_I

    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I

    mean_a = cv2.boxFilter(a, -1, ksize)
    mean_b = cv2.boxFilter(b, -1, ksize)

    q = mean_a * guide + mean_b
    return q

def estimate_atmospheric_light_L(R_float):
    dark_channel = cv2.erode(np.min(R_float, axis=2), np.ones((15, 15), np.uint8))
    num_pixels = dark_channel.size
    num_brightest = int(max(num_pixels * 0.001, 1))

    indices = np.argsort(dark_channel.flatten())[::-1][:num_brightest]
    R_flat = R_float.reshape(-1, 3)

    return np.mean(R_flat[indices], axis=0)

def get_distance_map_l(R_rgb, midas, transform, device, max_dist=100.0):
    input_batch = transform(R_rgb).to(device)

    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1), size=R_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    disparity_map = prediction.cpu().numpy()

    disparity_norm = (disparity_map - disparity_map.min()) / (disparity_map.max() - disparity_map.min() + 1e-8)
    disparity_clipped = np.clip(disparity_norm, 0.05, 1.0)

    distance_map = 1.0 / disparity_clipped
    distance_map = (distance_map - distance_map.min()) / (distance_map.max() - distance_map.min() + 1e-8)

    return distance_map * max_dist

def process_image_directory(input_dir, output_dir, max_dist=100.0, b_aero=0.015, aero_color=(0.75, 0.65, 0.45)):
    fog_levels = [0.01]

    # Create the main output directory
    os.makedirs(output_dir, exist_ok=True)

    # Find all images (jpg, png, jpeg) in the directory
    valid_extensions = ('*.jpg', '*.jpeg', '*.png')
    image_paths = []
    for ext in valid_extensions:
        image_paths.extend(glob.glob(os.path.join(input_dir, ext)))
        # Also check for uppercase extensions
        image_paths.extend(glob.glob(os.path.join(input_dir, ext.upper())))

    total_images = len(image_paths)
    if total_images == 0:
        print(f"No images found in {input_dir}")
        return

    print(f"\nFound {total_images} images. Loading MiDaS depth model...")

    # Load MiDaS ONCE globally to speed up batch processing
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small").to(device)
    midas.eval()
    transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform

    print("\n--- Starting Batch Process ---")

    # Process each image with enumeration for percentage tracking
    for idx, img_path in enumerate(image_paths, 1):
        img_name = os.path.basename(img_path)
        img_basename, img_ext = os.path.splitext(img_name)

        # Calculate integer percentage (1, 2, 3... 100)
        percent = int((idx / total_images) * 100)

        # Keep it at 1% minimum even for the first few images to look clean
        if percent == 0:
            percent = 1

        # --- NEW SKIP CHECK LOGIC ---
        # Define what the saved filename will be *before* processing
        save_name = f"{img_basename}_fog{fog_levels[0]}_aero{b_aero}{img_ext}"
        save_path = os.path.join(output_dir, save_name)

        # If the file already exists in the destination, skip all the heavy math!
        if os.path.exists(save_path):
            print(f"{percent}% - SKIPPING: {save_name} already exists.")
            continue
        # ----------------------------

        print(f"{percent}% - Processing: {img_name}")

        bgr_img = cv2.imread(img_path)
        if bgr_img is None:
            print(f"   -> ERROR: Could not read {img_name}, skipping.")
            continue

        R = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
        R_float = R.astype(np.float32) / 255.0
        R_gray = cv2.cvtColor(R, cv2.COLOR_RGB2GRAY)

        # Calculate L and Depth map ONLY ONCE per image
        L = estimate_atmospheric_light_L(R_float)
        l_x = get_distance_map_l(R, midas, transform, device, max_dist=max_dist)

        # Generate and save the variation directly into the output folder
        for b_fog in fog_levels:
            total_beta = b_fog + b_aero

            # Transmission
            t_hat = np.exp(-total_beta * l_x)
            t = guided_filter(R_gray, t_hat, radius=20, eps=1e-3)
            t = np.repeat(t[:, :, np.newaxis], 3, axis=2)

            # Effective Light (Fog + Aerosol color mix)
            L_eff = (b_fog * L + b_aero * np.array(aero_color)) / total_beta

            # Render
            I_x = R_float * t + L_eff * (1 - t)
            I_x_8bit = (np.clip(I_x, 0, 1) * 255).astype(np.uint8)

            # Convert back to BGR for OpenCV saving
            I_x_bgr = cv2.cvtColor(I_x_8bit, cv2.COLOR_RGB2BGR)

            # Save the file directly in the main output folder
            # (save_name and save_path were already defined above for the skip check)
            cv2.imwrite(save_path, I_x_bgr)

    print("\nBatch processing 100% complete! Check your Google Drive folder.")

# ==========================================
# Execution Block
# ==========================================
# Paths pointing strictly to your designated Google Drive folders
input_directory = "/content/drive/MyDrive/MES PROJECT GROUP 6/MES PROJECT Dataset/MES_DATASET"
output_directory = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Foggy_Original_Dataset"

# Run the batch processor with beta = 0.01 (defined inside the function)
process_image_directory(input_directory, output_directory, max_dist=100.0, b_aero=0.015)



Found 1118 images. Loading MiDaS depth model...


/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading weights:  None


/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt


100%|██████████| 81.8M/81.8M [00:01<00:00, 61.8MB/s]
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Processing: img_702.jpg...
   -> Saved directly to output folder for img_702.jpg
Processing: img_662.jpg...
   -> Saved directly to output folder for img_662.jpg
Processing: img_622.jpg...
   -> Saved directly to output folder for img_622.jpg
Processing: img_673.jpg...
   -> Saved directly to output folder for img_673.jpg
Processing: img_613.jpg...
   -> Saved directly to output folder for img_613.jpg
Processing: img_623.jpg...
   -> Saved directly to output folder for img_623.jpg
Processing: img_679.jpg...
   -> Saved directly to output folder for img_679.jpg
Processing: img_656.jpg...
   -> Saved directly to output folder for img_656.jpg
Processing: img_680.jpg...
   -> Saved directly to output folder for img_680.jpg
Processing: img_684.jpg...
   -> Saved directly to output folder for img_684.jpg
Processing: img_616.jpg...
   -> Saved directly to output folder for img_616.jpg
Processing: img_627.jpg...
   -> Saved directly to output folder for img_627.jpg
Processing: img_677.jpg...
 

In [ ]:
import cv2
import numpy as np
import torch
import os
import glob

def guided_filter(guide, src, radius, eps):
    guide = guide.astype(np.float32) / 255.0
    src = src.astype(np.float32)

    ksize = (2 * radius + 1, 2 * radius + 1)
    mean_I = cv2.boxFilter(guide, -1, ksize)
    mean_p = cv2.boxFilter(src, -1, ksize)
    mean_Ip = cv2.boxFilter(guide * src, -1, ksize)
    cov_Ip = mean_Ip - mean_I * mean_p

    mean_II = cv2.boxFilter(guide * guide, -1, ksize)
    var_I = mean_II - mean_I * mean_I

    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I

    mean_a = cv2.boxFilter(a, -1, ksize)
    mean_b = cv2.boxFilter(b, -1, ksize)

    q = mean_a * guide + mean_b
    return q

def estimate_atmospheric_light_L(R_float):
    dark_channel = cv2.erode(np.min(R_float, axis=2), np.ones((15, 15), np.uint8))
    num_pixels = dark_channel.size
    num_brightest = int(max(num_pixels * 0.001, 1))

    indices = np.argsort(dark_channel.flatten())[::-1][:num_brightest]
    R_flat = R_float.reshape(-1, 3)

    return np.mean(R_flat[indices], axis=0)

def get_distance_map_l(R_rgb, midas, transform, device, max_dist=100.0):
    input_batch = transform(R_rgb).to(device)

    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1), size=R_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    disparity_map = prediction.cpu().numpy()

    disparity_norm = (disparity_map - disparity_map.min()) / (disparity_map.max() - disparity_map.min() + 1e-8)
    disparity_clipped = np.clip(disparity_norm, 0.05, 1.0)

    distance_map = 1.0 / disparity_clipped
    distance_map = (distance_map - distance_map.min()) / (distance_map.max() - distance_map.min() + 1e-8)

    return distance_map * max_dist

def process_image_directory(input_dir, output_dir, max_dist=100.0, b_aero=0.015, aero_color=(0.75, 0.65, 0.45)):
    fog_levels = [0.01]

    # Create the main output directory
    os.makedirs(output_dir, exist_ok=True)

    # Find all images (jpg, png, jpeg) in the directory
    valid_extensions = ('*.jpg', '*.jpeg', '*.png')
    image_paths = []
    for ext in valid_extensions:
        image_paths.extend(glob.glob(os.path.join(input_dir, ext)))
        # Also check for uppercase extensions
        image_paths.extend(glob.glob(os.path.join(input_dir, ext.upper())))

    total_images = len(image_paths)
    if total_images == 0:
        print(f"No images found in {input_dir}")
        return

    print(f"\nFound {total_images} images. Loading MiDaS depth model...")

    # Load MiDaS ONCE globally to speed up batch processing
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small").to(device)
    midas.eval()
    transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform

    print("\n--- Starting Batch Process ---")

    # Process each image with enumeration for percentage tracking
    for idx, img_path in enumerate(image_paths, 1):
        img_name = os.path.basename(img_path)
        img_basename, img_ext = os.path.splitext(img_name)

        # Calculate integer percentage (1, 2, 3... 100)
        percent = int((idx / total_images) * 100)

        # Keep it at 1% minimum even for the first few images to look clean
        if percent == 0:
            percent = 1

        # --- NEW SKIP CHECK LOGIC ---
        # Define what the saved filename will be *before* processing
        save_name = f"{img_basename}_fog{fog_levels[0]}_aero{b_aero}{img_ext}"
        save_path = os.path.join(output_dir, save_name)

        # If the file already exists in the destination, skip all the heavy math!
        if os.path.exists(save_path):
            print(f"{percent}% - SKIPPING: {save_name} already exists.")
            continue
        # ----------------------------

        print(f"{percent}% - Processing: {img_name}")

        bgr_img = cv2.imread(img_path)
        if bgr_img is None:
            print(f"   -> ERROR: Could not read {img_name}, skipping.")
            continue

        R = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
        R_float = R.astype(np.float32) / 255.0
        R_gray = cv2.cvtColor(R, cv2.COLOR_RGB2GRAY)

        # Calculate L and Depth map ONLY ONCE per image
        L = estimate_atmospheric_light_L(R_float)
        l_x = get_distance_map_l(R, midas, transform, device, max_dist=max_dist)

        # Generate and save the variation directly into the output folder
        for b_fog in fog_levels:
            total_beta = b_fog + b_aero

            # Transmission
            t_hat = np.exp(-total_beta * l_x)
            t = guided_filter(R_gray, t_hat, radius=20, eps=1e-3)
            t = np.repeat(t[:, :, np.newaxis], 3, axis=2)

            # Effective Light (Fog + Aerosol color mix)
            L_eff = (b_fog * L + b_aero * np.array(aero_color)) / total_beta

            # Render
            I_x = R_float * t + L_eff * (1 - t)
            I_x_8bit = (np.clip(I_x, 0, 1) * 255).astype(np.uint8)

            # Convert back to BGR for OpenCV saving
            I_x_bgr = cv2.cvtColor(I_x_8bit, cv2.COLOR_RGB2BGR)

            # Save the file directly in the main output folder
            # (save_name and save_path were already defined above for the skip check)
            cv2.imwrite(save_path, I_x_bgr)

    print("\nBatch processing 100% complete! Check your Google Drive folder.")

# ==========================================
# Execution Block
# ==========================================
# Paths pointing strictly to your designated Google Drive folders
input_directory = "/content/drive/MyDrive/MES PROJECT GROUP 6/MES PROJECT Dataset/MES_DATASET_aug"
output_directory = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Foggy_Augmented_Dataset"

# Run the batch processor with beta = 0.01 (defined inside the function)
process_image_directory(input_directory, output_directory, max_dist=100.0, b_aero=0.015)


Found 3354 images. Loading MiDaS depth model...


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Loading weights:  None


Using cache found in /root/.cache/torch/hub/rwightman_gen-efficientnet-pytorch_master
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master



--- Starting Batch Process ---
1% - SKIPPING: aug_2_img_227_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_0_img_47_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_1_img_47_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_2_img_47_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_0_img_232_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_1_img_232_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_2_img_232_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_0_img_236_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_1_img_236_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_2_img_236_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_0_img_39_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_1_img_39_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_2_img_39_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_0_img_237_fog0.01_aero0.015.jpg already exists.
1% - SKIPPING: aug_1_img_237_fog0.01

# Velodyne generation

In [ ]:
import torch
import cv2
import numpy as np
import os
from tqdm import tqdm # This gives us a nice progress bar!

# --- 1. Define Directories ---
# REPLACE these with your actual Drive paths
INPUT_DIR = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/Foggy_Original_Dataset"
OUTPUT_DIR = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/Velodyne"

# Create the output folder if it doesn't already exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. Setup Model and Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load MiDaS
print("Loading MiDaS model...")
midas = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas.to(device)
midas.eval()

transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = transforms.dpt_transform

# --- 3. Gather Images ---
print("Scanning input directory...")
all_files = os.listdir(INPUT_DIR)

# Filter out hidden files and only keep images
valid_images = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if not valid_images:
    print(f"Error: No valid images found in {INPUT_DIR}!")
    exit()

print(f"Found {len(valid_images)} images. Starting batch processing...")

# --- 4. Loop Through Every Image ---
for img_name in tqdm(valid_images):

    # Safely extract the filename without the .jpg extension
    base_name = os.path.splitext(img_name)[0]

    # Define the exact output path for this specific .bin file
    bin_output_path = os.path.join(OUTPUT_DIR, f"{base_name}.bin")

    # SKIP if this file was already processed (great if the notebook crashes and you need to restart)
    if os.path.exists(bin_output_path):
        continue

    # Load Image
    img_path = os.path.join(INPUT_DIR, img_name)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not read {img_name}, skipping.")
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    # Camera Intrinsics
    fx, fy = 700.0, 700.0
    cx, cy = w / 2.0, h / 2.0

    # Run Inference
    input_batch = transform(img_rgb).to(device)

    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=(h, w), # Use exact dimensions
            mode="bicubic",
            align_corners=False
        ).squeeze()

    # Move prediction to CPU and normalize
    depth = prediction.cpu().numpy()
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

    # Back-project to 3D Point Cloud
    u, v = np.meshgrid(np.arange(w), np.arange(h))
    Z = depth * 50
    X = (u - cx) * Z / fx
    Y = (v - cy) * Z / fy

    # Stack and filter
    points = np.stack((X, Y, Z), axis=-1).reshape(-1, 3)
    points = points[(points[:, 2] > 1.5) & (points[:, 2] < 60)]

    # Downsample
    keep = int(len(points) * 0.03)
    if keep > 0:
        points = points[np.random.choice(len(points), keep, replace=False)]

    # Add intensity and save
    intensity = np.random.uniform(0.2, 1.0, (points.shape[0], 1))
    velodyne_data = np.concatenate((points, intensity), axis=1).astype(np.float32)
    velodyne_data.tofile(bin_output_path)

print(f"\n✅ All done! {len(valid_images)} files successfully converted and saved to {OUTPUT_DIR}")

Using device: cuda
Loading MiDaS model...


/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(


Downloading: "https://github.com/isl-org/MiDaS/releases/download/v3/dpt_hybrid_384.pt" to /root/.cache/torch/hub/checkpoints/dpt_hybrid_384.pt


100%|██████████| 470M/470M [00:04<00:00, 103MB/s] 
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Scanning input directory...
Found 1118 images. Starting batch processing...


100%|██████████| 1118/1118 [15:01<00:00,  1.24it/s]


✅ All done! 1118 files successfully converted and saved to /content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/Velodyne


In [ ]:
import torch
import cv2
import numpy as np
import os
from tqdm import tqdm # This gives us a nice progress bar!

# --- 1. Define Directories ---
# REPLACE these with your actual Drive paths
INPUT_DIR = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/Foggy_Original_Dataset"
OUTPUT_DIR = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/Velodyne"

# Create the output folder if it doesn't already exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. Setup Model and Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load MiDaS
print("Loading MiDaS model...")
midas = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas.to(device)
midas.eval()

transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = transforms.dpt_transform

# --- 3. Gather Images ---
print("Scanning input directory...")
all_files = os.listdir(INPUT_DIR)

# Filter out hidden files and only keep images
valid_images = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if not valid_images:
    print(f"Error: No valid images found in {INPUT_DIR}!")
    exit()

print(f"Found {len(valid_images)} images. Starting batch processing...")

# --- 4. Loop Through Every Image ---
for img_name in tqdm(valid_images):

    # Safely extract the filename without the .jpg extension
    base_name = os.path.splitext(img_name)[0]

    # Define the exact output path for this specific .bin file
    bin_output_path = os.path.join(OUTPUT_DIR, f"{base_name}.bin")

    # SKIP if this file was already processed (great if the notebook crashes and you need to restart)
    if os.path.exists(bin_output_path):
        continue

    # Load Image
    img_path = os.path.join(INPUT_DIR, img_name)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not read {img_name}, skipping.")
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    # Camera Intrinsics
    fx, fy = 700.0, 700.0
    cx, cy = w / 2.0, h / 2.0

    # Run Inference
    input_batch = transform(img_rgb).to(device)

    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=(h, w), # Use exact dimensions
            mode="bicubic",
            align_corners=False
        ).squeeze()

    # Move prediction to CPU and normalize
    depth = prediction.cpu().numpy()
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

    # Back-project to 3D Point Cloud
    u, v = np.meshgrid(np.arange(w), np.arange(h))
    Z = depth * 50
    X = (u - cx) * Z / fx
    Y = (v - cy) * Z / fy

    # Stack and filter
    points = np.stack((X, Y, Z), axis=-1).reshape(-1, 3)
    points = points[(points[:, 2] > 1.5) & (points[:, 2] < 60)]

    # Downsample
    keep = int(len(points) * 0.03)
    if keep > 0:
        points = points[np.random.choice(len(points), keep, replace=False)]

    # Add intensity and save
    intensity = np.random.uniform(0.2, 1.0, (points.shape[0], 1))
    velodyne_data = np.concatenate((points, intensity), axis=1).astype(np.float32)
    velodyne_data.tofile(bin_output_path)

print(f"\n✅ All done! {len(valid_images)} files successfully converted and saved to {OUTPUT_DIR}")

# GENERATION OF LEBELING

Original Images

In [ ]:
import numpy as np
import cv2
import os
import torch
import re
from ultralytics import YOLO
from tqdm import tqdm

img_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/MES PROJECT Dataset/MES_DATASET"
label_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/training/label_2"

os.makedirs(label_folder, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading models...")

midas = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid").to(device).eval()
transform = torch.hub.load("intel-isl/MiDaS", "transforms").dpt_transform

model = YOLO("yolov8m.pt")

# STRICT KITTI CLASSES ONLY
coco_to_kitti = {
    0: "Pedestrian",
    2: "Car",
    1: "Cyclist",
    3: "Cyclist",   # motorcycle → cyclist (optional)
    5: "Car",       # bus → car (merge safely)
    7: "Car"        # truck → car (merge safely)
}

fx, fy = 700.0, 700.0
cx, cy = 640.0, 360.0

batch_size = 8

images = sorted([
    f for f in os.listdir(img_folder)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

print(f"Found {len(images)} images")

for start in tqdm(range(0, len(images), batch_size)):
    batch = images[start:start+batch_size]
    paths = [os.path.join(img_folder, img) for img in batch]

    results = model(paths, verbose=False)

    for img_name, result in zip(batch, results):

        match = re.search(r'(\d+)', img_name)
        base_name = f"{int(match.group(1)):06d}" if match else img_name.split('.')[0]

        label_file = os.path.join(label_folder, f"{base_name}.txt")

        img = cv2.imread(os.path.join(img_folder, img_name))
        if img is None:
            continue

        h_img, w_img, _ = img.shape
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        input_batch = transform(img_rgb).to(device)

        with torch.no_grad():
            prediction = midas(input_batch)
            prediction = torch.nn.functional.interpolate(
                prediction.unsqueeze(1),
                size=img_rgb.shape[:2],
                mode="bicubic",
                align_corners=False
            ).squeeze()

        depth = prediction.cpu().numpy()
        depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        lines = []

        for box, cls in zip(boxes, classes):
            cls = int(cls)

            if cls not in coco_to_kitti:
                continue

            x1, y1, x2, y2 = map(int, box)

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w_img - 1, x2), min(h_img - 1, y2)

            if x2 <= x1 or y2 <= y1:
                continue

            crop = depth[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            # safer depth estimate
            z = np.percentile(crop, 60) * 40 + 5.0

            xc = ((x1 + x2) / 2 - cx) * z / fx
            yc = ((y1 + y2) / 2 - cy) * z / fy

            obj_type = coco_to_kitti[cls]

            lines.append(
                f"{obj_type} 0 0 0 "
                f"{x1:.2f} {y1:.2f} {x2:.2f} {y2:.2f} "
                f"1.50 1.60 3.70 "
                f"{xc:.2f} {yc:.2f} {z:.2f} 0.00"
            )

        # ✅ only write file if objects exist
        if len(lines) > 0:
            with open(label_file, "w") as f:
                f.write("\n".join(lines))

print("DONE → KITTI labels ready for OpenPCDet 🚀")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading models...


/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip


/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(


Downloading: "https://github.com/isl-org/MiDaS/releases/download/v3/dpt_hybrid_384.pt" to /root/.cache/torch/hub/checkpoints/dpt_hybrid_384.pt


100%|██████████| 470M/470M [00:10<00:00, 47.8MB/s]
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Found 3354 images


100%|██████████| 420/420 [25:09<00:00,  3.59s/it]

DONE → KITTI labels ready for OpenPCDet 🚀


Statics

In [ ]:
import os
from collections import defaultdict

# track counts
obj_counts = defaultdict(int)
img_counts = defaultdict(int)

total_images = 0
total_objects = 0

# my label folder path
label_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/training/label_2"

# go through all the txt files
for file in os.listdir(label_folder):
    if not file.endswith(".txt"):
        continue

    total_images += 1
    path = os.path.join(label_folder, file)

    with open(path) as f:
        lines = f.readlines()

    classes_in_image = set()

    for line in lines:
        parts = line.strip().split()

        # kitti format has 15 columns, so skip any weird lines
        if len(parts) < 15:
            continue

        cls = parts[0]  # get the class name (Car, Pedestrian, etc.)
        obj_counts[cls] += 1
        total_objects += 1
        classes_in_image.add(cls)

    for cls in classes_in_image:
        img_counts[cls] += 1

# print out the results
print("--- Dataset Stats ---")
print(f"Total images: {total_images}")
print(f"Total objects: {total_objects}")

if total_images > 0:
    print(f"Average objects per image: {round(total_objects / total_images, 2)}")

print("\nCounts per class:")
# sort by highest count first
for cls, count in sorted(obj_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"- {cls}: {count}")

--- Dataset Stats ---
Total images: 1118
Total objects: 7584
Average objects per image: 6.78

Counts per class:
- Pedestrian: 3220
- Car: 1673
- Truck: 1077
- Motorcycle: 1012
- Bus: 327
- Bicycle: 232
- Animal: 43


Augumented Data

In [ ]:
import numpy as np
import cv2
import os
import torch
import re
from ultralytics import YOLO
from tqdm import tqdm

img_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/MES PROJECT Dataset/MES_DATASET_aug"
label_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Augmented/training/label_2"

os.makedirs(label_folder, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading models...")

midas = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid").to(device).eval()
transform = torch.hub.load("intel-isl/MiDaS", "transforms").dpt_transform

model = YOLO("yolov8m.pt")

# ✅ STRICT KITTI CLASSES ONLY
coco_to_kitti = {
    0: "Pedestrian",
    2: "Car",
    1: "Cyclist",
    3: "Cyclist",   # motorcycle → cyclist (optional)
    5: "Car",       # bus → car (merge safely)
    7: "Car"        # truck → car (merge safely)
}

fx, fy = 700.0, 700.0
cx, cy = 640.0, 360.0

batch_size = 8

images = sorted([
    f for f in os.listdir(img_folder)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

print(f"Found {len(images)} images")

for start in tqdm(range(0, len(images), batch_size)):
    batch = images[start:start+batch_size]
    paths = [os.path.join(img_folder, img) for img in batch]

    results = model(paths, verbose=False)

    for img_name, result in zip(batch, results):

        match = re.search(r'(\d+)', img_name)
        base_name = f"{int(match.group(1)):06d}" if match else img_name.split('.')[0]

        label_file = os.path.join(label_folder, f"{base_name}.txt")

        img = cv2.imread(os.path.join(img_folder, img_name))
        if img is None:
            continue

        h_img, w_img, _ = img.shape
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        input_batch = transform(img_rgb).to(device)

        with torch.no_grad():
            prediction = midas(input_batch)
            prediction = torch.nn.functional.interpolate(
                prediction.unsqueeze(1),
                size=img_rgb.shape[:2],
                mode="bicubic",
                align_corners=False
            ).squeeze()

        depth = prediction.cpu().numpy()
        depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        lines = []

        for box, cls in zip(boxes, classes):
            cls = int(cls)

            if cls not in coco_to_kitti:
                continue

            x1, y1, x2, y2 = map(int, box)

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w_img - 1, x2), min(h_img - 1, y2)

            if x2 <= x1 or y2 <= y1:
                continue

            crop = depth[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            # safer depth estimate
            z = np.percentile(crop, 60) * 40 + 5.0

            xc = ((x1 + x2) / 2 - cx) * z / fx
            yc = ((y1 + y2) / 2 - cy) * z / fy

            obj_type = coco_to_kitti[cls]

            lines.append(
                f"{obj_type} 0 0 0 "
                f"{x1:.2f} {y1:.2f} {x2:.2f} {y2:.2f} "
                f"1.50 1.60 3.70 "
                f"{xc:.2f} {yc:.2f} {z:.2f} 0.00"
            )

        # ✅ only write file if objects exist
        if len(lines) > 0:
            with open(label_file, "w") as f:
                f.write("\n".join(lines))

print("DONE → KITTI labels ready for OpenPCDet 🚀")

Loading models...


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Found 3354 images


100%|██████████| 420/420 [23:06<00:00,  3.30s/it]

DONE → KITTI labels ready for OpenPCDet 🚀


In [ ]:
import os
from collections import defaultdict

# track counts
obj_counts = defaultdict(int)
img_counts = defaultdict(int)

total_images = 0
total_objects = 0

# my label folder path
label_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Augmented/training/label_2"

# go through all the txt files
for file in os.listdir(label_folder):
    if not file.endswith(".txt"):
        continue

    total_images += 1
    path = os.path.join(label_folder, file)

    with open(path) as f:
        lines = f.readlines()

    classes_in_image = set()

    for line in lines:
        parts = line.strip().split()

        # kitti format has 15 columns, so skip any weird lines
        if len(parts) < 15:
            continue

        cls = parts[0]  # get the class name (Car, Pedestrian, etc.)
        obj_counts[cls] += 1
        total_objects += 1
        classes_in_image.add(cls)

    for cls in classes_in_image:
        img_counts[cls] += 1

# print out the results
print("--- Dataset Stats ---")
print(f"Total images: {total_images}")
print(f"Total objects: {total_objects}")

if total_images > 0:
    print(f"Average objects per image: {round(total_objects / total_images, 2)}")

print("\nCounts per class:")
# sort by highest count first
for cls, count in sorted(obj_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"- {cls}: {count}")


LABEL STATISTICS
Total Images Processed: 3354
Total Objects Found: 18914
Average Objects per Image: 5.64

OBJECT COUNTS PER CLASS:
  Pedestrian: 7701
  Car: 4721
  Truck: 2642
  Motorcycle: 2273
  Bus: 930
  Bicycle: 563
  Animal: 84



# GENERATION OF CALIBERATION FILES

In [ ]:
import os
from tqdm import tqdm

# my folders for the dataset
img_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/training/image_2"
calib_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/training/calib"

# create calib folder if I don't have it yet
if not os.path.exists(calib_folder):
    os.makedirs(calib_folder)

# camera values (must match the ones from phase 4)
fx, fy = 700.0, 700.0
cx, cy = 640.0, 360.0

# get all image files
files = os.listdir(img_folder)
images = sorted([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

new_files = 0
skipped = 0

if len(images) == 0:
    print("Warning: No images found. Check the path.")
else:
    print(f"Found {len(images)} images. Generating calib files...")

    for img_name in tqdm(images):
        # just get the name without .jpg or .png
        base_name = img_name.split('.')[0]
        calib_file = os.path.join(calib_folder, f"{base_name}.txt")

        # if the file is already there, skip it
        if os.path.exists(calib_file):
            skipped += 1
            continue

        with open(calib_file, "w") as f:
            # camera projection matrices
            f.write(f"P0: {fx} 0 {cx} 0 0 {fy} {cy} 0 0 0 1 0\n")
            f.write(f"P1: {fx} 0 {cx} 0 0 {fy} {cy} 0 0 0 1 0\n")
            f.write(f"P2: {fx} 0 {cx} 0 0 {fy} {cy} 0 0 0 1 0\n")
            f.write(f"P3: {fx} 0 {cx} 0 0 {fy} {cy} 0 0 0 1 0\n")

            # identity matrices for the pseudo-lidar setup
            f.write("R0_rect: 1 0 0 0 1 0 0 0 1\n")
            f.write("Tr_velo_to_cam: 1 0 0 0 0 1 0 0 0 0 1 0\n")
            f.write("Tr_imu_to_velo: 1 0 0 0 0 1 0 0 0 0 1 0\n")

        new_files += 1

    # check how many files are actually in the folder now
    total_calibs = len([f for f in os.listdir(calib_folder) if f.endswith('.txt')])

    print("\nDone!")
    print(f"Images scanned: {len(images)}")
    print(f"Generated: {new_files}")
    print(f"Skipped: {skipped}")
    print(f"Total calibs ready: {total_calibs}")

Generating Original Calibs: 100%|██████████| 1118/1118 [00:12<00:00, 86.08it/s]


✅ CALIBRATION GENERATION COMPLETE
Total Input Images Found:   1118
Newly Generated Calibs:     1118
Already Existing (Skipped): 0
------------------------------------------
Total Calibs in Folder:     1118


In [ ]:
import os
from tqdm import tqdm

img_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Augmented/training/image_2"
calib_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Augmented/training/calib"

# create calib folder if I don't have it yet
if not os.path.exists(calib_folder):
    os.makedirs(calib_folder)

# camera values (must match the ones from phase 4)
fx, fy = 700.0, 700.0
cx, cy = 640.0, 360.0

# get all image files
files = os.listdir(img_folder)
images = sorted([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

new_files = 0
skipped = 0

if len(images) == 0:
    print("Warning: No images found. Check the path.")
else:
    print(f"Found {len(images)} images. Generating calib files...")

    for img_name in tqdm(images):
        # just get the name without .jpg or .png
        base_name = img_name.split('.')[0]
        calib_file = os.path.join(calib_folder, f"{base_name}.txt")

        # if the file is already there, skip it
        if os.path.exists(calib_file):
            skipped += 1
            continue

        with open(calib_file, "w") as f:
            # camera projection matrices
            f.write(f"P0: {fx} 0 {cx} 0 0 {fy} {cy} 0 0 0 1 0\n")
            f.write(f"P1: {fx} 0 {cx} 0 0 {fy} {cy} 0 0 0 1 0\n")
            f.write(f"P2: {fx} 0 {cx} 0 0 {fy} {cy} 0 0 0 1 0\n")
            f.write(f"P3: {fx} 0 {cx} 0 0 {fy} {cy} 0 0 0 1 0\n")

            # identity matrices for the pseudo-lidar setup
            f.write("R0_rect: 1 0 0 0 1 0 0 0 1\n")
            f.write("Tr_velo_to_cam: 1 0 0 0 0 1 0 0 0 0 1 0\n")
            f.write("Tr_imu_to_velo: 1 0 0 0 0 1 0 0 0 0 1 0\n")

        new_files += 1

    # check how many files are actually in the folder now
    total_calibs = len([f for f in os.listdir(calib_folder) if f.endswith('.txt')])

    print("\nDone!")
    print(f"Images scanned: {len(images)}")
    print(f"Generated: {new_files}")
    print(f"Skipped: {skipped}")
    print(f"Total calibs ready: {total_calibs}")

Found 3354 images. Generating calib files...


100%|██████████| 3354/3354 [00:00<00:00, 4052.86it/s]



Done!
Images scanned: 3354
Generated: 0
Skipped: 3354
Total calibs ready: 3354


# GENERATION OF IMAGESETS


In [ ]:
import os
import random

# My drive paths for the dataset
img_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/training/image_2"
output_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/ImageSets"

# make sure output folder exists before saving
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# get all the image files from the folder
all_files = os.listdir(img_folder)
images = [f for f in all_files if f.endswith('.png') or f.endswith('.jpg') or f.endswith('.jpeg')]
images.sort()

# get just the file names without extensions (like 020231)
names = [img.split('.')[0] for img in images]

if len(names) == 0:
    print("Error: No images found. Check the folder path.")
else:
    # shuffle the dataset
    # keeping seed 42 so the split stays the same if I run this cell again
    random.seed(42)
    random.shuffle(names)

    total = len(names)
    print("Total files found:", total)

    # 80-10-10 split
    train_count = int(total * 0.8)
    val_count = int(total * 0.1)

    train_set = names[:train_count]
    val_set = names[train_count : train_count + val_count]
    test_set = names[train_count + val_count:]

    # helper function to save the text files
    def save_split(filename, data):
        path = os.path.join(output_folder, filename)
        with open(path, 'w') as f:
            for item in data:
                f.write(item + '\n')

    # save everything to the ImageSets folder
    save_split('train.txt', train_set)
    save_split('val.txt', val_set)
    save_split('test.txt', test_set)

    # combining train and val for the final training phase later
    trainval_set = train_set + val_set
    save_split('trainval.txt', trainval_set)

    print("Done splitting the dataset!")
    print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

Total files found: 1118
Done splitting the dataset!
Train: 894 | Val: 111 | Test: 113


In [ ]:
import os
import random

# My drive paths for the dataset
img_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Augmented/training/image_2"
output_folder = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Augmented/ImageSets"

# make sure output folder exists before saving
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# get all the image files from the folder
all_files = os.listdir(img_folder)
images = [f for f in all_files if f.endswith('.png') or f.endswith('.jpg') or f.endswith('.jpeg')]
images.sort()

# get just the file names without extensions (like 020231)
names = [img.split('.')[0] for img in images]

if len(names) == 0:
    print("Error: No images found. Check the folder path.")
else:
    # shuffle the dataset
    # keeping seed 42 so the split stays the same if I run this cell again
    random.seed(42)
    random.shuffle(names)

    total = len(names)
    print("Total files found:", total)

    # 80-10-10 split
    train_count = int(total * 0.8)
    val_count = int(total * 0.1)

    train_set = names[:train_count]
    val_set = names[train_count : train_count + val_count]
    test_set = names[train_count + val_count:]

    # helper function to save the text files
    def save_split(filename, data):
        path = os.path.join(output_folder, filename)
        with open(path, 'w') as f:
            for item in data:
                f.write(item + '\n')

    # save everything to the ImageSets folder
    save_split('train.txt', train_set)
    save_split('val.txt', val_set)
    save_split('test.txt', test_set)

    # combining train and val for the final training phase later
    trainval_set = train_set + val_set
    save_split('trainval.txt', trainval_set)

    print("Done splitting the dataset!")
    print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

Total files found: 3354
Done splitting the dataset!
Train: 2683 | Val: 335 | Test: 336


# INFOS FILE GENERATION

In [ ]:
%%bash
cd /content/OpenPCDet

# Set your base path
BASE_DIR="/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti"

# ==========================================
# 1. Generate Infos for the ORIGINAL Dataset
# ==========================================
echo "🔗 Linking Original Dataset..."
ln -sfn "$BASE_DIR/Dataset_Original" data/kitti

echo "⚙️ Generating infos and gt_database for Original Dataset..."
python -m pcdet.datasets.kitti.kitti_dataset create_kitti_infos tools/cfgs/dataset_configs/kitti_dataset.yaml

echo "✅ Original Dataset processing complete!"
echo "---------------------------------------------------"

# ==========================================
# 2. Generate Infos for the AUGMENTED Dataset
# ==========================================
echo "🔗 Linking Augmented Dataset..."
ln -sfn "$BASE_DIR/Dataset_Augmented" data/kitti

echo "⚙️ Generating infos and gt_database for Augmented Dataset..."
python -m pcdet.datasets.kitti.kitti_dataset create_kitti_infos tools/cfgs/dataset_configs/kitti_dataset.yaml

echo "✅ Augmented Dataset processing complete!"

🔗 Linking Original Dataset...
⚙️ Generating infos and gt_database for Original Dataset...
[1/40] [GCC][c++/pch]/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h.gch
FAILED: [code=1] /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h.gch 
g++ -MMD -MT /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h.gch -MF /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h.gch.d -I "/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include" -I "/usr/local/lib/python3.12/dist-packages/pybind11/include" -I "/usr/include/python3.12" -I "/usr/local/cuda/include" -I "/usr/local/lib/python3.12/dist-packages/include" -O3 -DTV_STATIC_VARIABLE_IMPLEMENTATION -std=c++14 -fPIC -x c++-header -DTV_ENABLE_HARDWARE_ACC -c /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorVie

<frozen runpy>:128: RuntimeWarning: 'pcdet.datasets.kitti.kitti_dataset' found in sys.modules after import of package 'pcdet.datasets.kitti', but prior to execution of 'pcdet.datasets.kitti.kitti_dataset'; this may result in unpredictable behaviour
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/OpenPCDet/pcdet/datasets/kitti/kitti_dataset.py", line 479, in <module>
    create_kitti_infos(
  File "/content/OpenPCDet/pcdet/datasets/kitti/kitti_dataset.py", line 443, in create_kitti_infos
    kitti_infos_train = dataset.get_infos(num_workers=workers, has_label=True, count_inside_pts=True)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/OpenPCDet/pcdet/datasets/kitti/kitti_dataset.py", line 222, in get_infos
    return list(infos)
           ^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", l

In [ ]:
ls "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/training/image_2" | head -n 5

0001.jpg
0002.jpg
0003.jpg
0004.jpg
0005.jpg


# GENERATION OF GT Database

In [ ]:
import os
import pickle
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. SET EXACT BASE DIRECTORY
# ==========================================
BASE_DIR = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti"

# ==========================================
# 2. DEFINING CONFIGURATIONS
# ==========================================
DATASETS = [
    {
        "prefix": "original",
        "info_path": "11_original_infos/original_infos_train.pkl",
        "gt_db_dir": "13_original_gt_database",
        "dbinfo_out": "11_original_infos/original_kitti_dbinfos_train.pkl"
    },
    {
        "prefix": "augmented",
        "info_path": "12_augmented_infos/augmented_infos_train.pkl",
        "gt_db_dir": "14_augmented_gt_database",
        "dbinfo_out": "12_augmented_infos/augmented_kitti_dbinfos_train.pkl"
    }
]

# ==========================================
# 3. 3D MATH: POINT-IN-BOX FILTER
# ==========================================
def get_points_in_box(pc_velo, loc, dims, ry, calib):
    """Filters point cloud to only keep points inside the 3D KITTI box."""
    h, w, l = dims  # KITTI format: height(y), width(x), length(z)
    cx, cy, cz = loc

    # 1. Convert Velodyne points to Camera Rectified coordinates
    pts = pc_velo[:, :3]
    pts_hom = np.hstack((pts, np.ones((pts.shape[0], 1))))
    pts_cam = pts_hom @ calib['Tr_velo_to_cam'].T
    pts_rect = pts_cam @ calib['R0_rect'].T

    # 2. Translate points to the box center
    pts_local = pts_rect - np.array([cx, cy, cz])

    # 3. Apply Inverse Rotation (around Camera Y axis)
    c, s = np.cos(-ry), np.sin(-ry)
    R_inv = np.array([
        [ c, 0, s],
        [ 0, 1, 0],
        [-s, 0, c]
    ])
    pts_local = pts_local @ R_inv.T

    # 4. Filter points inside the box boundaries
    # KITTI Box center is at the BOTTOM of the object
    # Camera coords: x=right, y=down, z=forward
    mask = (pts_local[:, 0] >= -w/2) & (pts_local[:, 0] <= w/2) & \
           (pts_local[:, 1] >= -h)   & (pts_local[:, 1] <= 0) & \
           (pts_local[:, 2] >= -l/2) & (pts_local[:, 2] <= l/2)

    return mask

# ==========================================
# 4. BUILD DATABASE FUNCTION
# ==========================================
def create_gt_database(config):
    info_file_path = os.path.join(BASE_DIR, config['info_path'])
    if not os.path.exists(info_file_path):
        print(f" Cannot find {info_file_path}. Skipping.")
        return

    # Load the training infos we generated in Phase 8
    with open(info_file_path, 'rb') as f:
        infos = pickle.load(f)

    gt_db_path = os.path.join(BASE_DIR, config['gt_db_dir'])
    os.makedirs(gt_db_path, exist_ok=True)

    all_db_infos = {}

    for info in tqdm(infos, desc=f"Building {config['prefix'].upper()} GT Database"):
        img_idx = info['image']['image_idx']

        # Load the full point cloud
        velo_rel_path = info['point_cloud']['velodyne_path']
        velo_full_path = os.path.join(BASE_DIR, velo_rel_path)
        if not os.path.exists(velo_full_path):
            continue

        pc_velo = np.fromfile(velo_full_path, dtype=np.float32).reshape(-1, 4)
        calib = info['calib']

        if 'annos' not in info:
            continue

        annos = info['annos']
        names = annos['name']

        # Loop through every object in the frame
        for i in range(len(names)):
            obj_name = names[i]
            if obj_name == 'DontCare':
                continue

            loc = annos['location'][i]
            dims = annos['dimensions'][i]
            ry = annos['rotation_y'][i]

            # Find which points belong to this specific object
            mask = get_points_in_box(pc_velo, loc, dims, ry, calib)
            pts_in_box = pc_velo[mask]
            num_points = pts_in_box.shape[0]

            # If the box is empty (heavy occlusion/far away), skip it
            if num_points == 0:
                continue

            # Save the cropped points
            filename = f"{img_idx}_{obj_name}_{i}.bin"
            filepath = os.path.join(gt_db_path, filename)
            pts_in_box.tofile(filepath)

            # Record metadata for the dbinfos dictionary
            db_info = {
                'name': obj_name,
                'path': f"{config['gt_db_dir']}/{filename}",
                'image_idx': img_idx,
                'gt_idx': i,
                'box3d_camera': np.concatenate([loc, dims, [ry]]),
                'num_points_in_gt': num_points,
                'difficulty': annos['occluded'][i] # Using occlusion as base difficulty
            }

            if obj_name not in all_db_infos:
                all_db_infos[obj_name] = []
            all_db_infos[obj_name].append(db_info)

    # Save the master dictionary to a .pkl file
    dbinfo_out_path = os.path.join(BASE_DIR, config['dbinfo_out'])
    with open(dbinfo_out_path, 'wb') as f:
        pickle.dump(all_db_infos, f)

    print(f" Finished {config['prefix']}. Saved dbinfos to {config['dbinfo_out']}")

# ==========================================
# 5. EXECUTE
# ==========================================
print("\n Starting GT Database Generation (This involves 3D math and may take a few minutes)...\n")
for dataset_config in DATASETS:
    create_gt_database(dataset_config)
    print("-" * 50)
print("\n ALL DONE! Your dataset is now 100% physically prepared.")


🚀 Starting GT Database Generation (This involves 3D math and may take a few minutes)...



Building ORIGINAL GT Database: 100%|██████████| 894/894 [11:34<00:00,  1.29it/s]


✅ Finished original. Saved dbinfos to 11_original_infos/original_kitti_dbinfos_train.pkl
--------------------------------------------------


Building AUGMENTED GT Database: 100%|██████████| 2683/2683 [30:15<00:00,  1.48it/s]

✅ Finished augmented. Saved dbinfos to 12_augmented_infos/augmented_kitti_dbinfos_train.pkl
--------------------------------------------------

🎉 ALL DONE! Your dataset is now 100% physically prepared.


# TRAINING THE ORIGINAL DATASET

In [ ]:
import os

# 1. Force Colab to create the folders if they are missing
os.makedirs('/content/OpenPCDet/tools/cfgs/kitti_models', exist_ok=True)

# 2. Define the 6-class configuration
yaml_content = """CLASS_NAMES: ['Car', 'Pedestrian', 'Bicycle', 'Motorcycle', 'Bus', 'Truck']

DATA_CONFIG:
    _BASE_CONFIG_: cfgs/dataset_configs/kitti_dataset.yaml

    INFO_PATH: {
        'train': ['11_original_infos/original_infos_train.pkl'],
        'test': ['11_original_infos/original_infos_val.pkl'],
    }

    DATA_AUGMENTATOR:
        DISABLE_AUG_LIST: ['placeholder']
        AUG_CONFIG_LIST:
            - NAME: gt_sampling
              USE_ROAD_PLANE: False
              DB_INFO_PATH:
                  - '11_original_infos/original_kitti_dbinfos_train.pkl'
              PREPARE: {
                 filter_by_min_points: ['Car:5', 'Pedestrian:5', 'Bicycle:5', 'Motorcycle:5', 'Bus:5', 'Truck:5'],
                 filter_by_difficulty: [-1],
              }
              SAMPLE_GROUPS: ['Car:15', 'Pedestrian:10', 'Bicycle:10', 'Motorcycle:10', 'Bus:4', 'Truck:4']
              NUM_POINT_FEATURES: 4
              DATABASE_WITH_FAKELIDAR: False
              REMOVE_EXTRA_WIDTH: [0.0, 0.0, 0.0]
              LIMIT_WHOLE_SCENE: True

MODEL:
    _BASE_CONFIG_: cfgs/kitti_models/second.yaml
    DENSE_HEAD:
        ANCHOR_GENERATOR_CONFIG:
            - {
                'class_name': 'Car',
                'anchor_sizes': [[3.9, 1.6, 1.56]],
                'anchor_rotations': [0, 1.57],
                'anchor_bottom_heights': [-1.78],
                'align_center': False,
                'feature_map_stride': 8,
                'matched_threshold': 0.6,
                'unmatched_threshold': 0.45
            }
            - {
                'class_name': 'Pedestrian',
                'anchor_sizes': [[0.8, 0.6, 1.73]],
                'anchor_rotations': [0, 1.57],
                'anchor_bottom_heights': [-0.6],
                'align_center': False,
                'feature_map_stride': 8,
                'matched_threshold': 0.5,
                'unmatched_threshold': 0.35
            }
            - {
                'class_name': 'Bicycle',
                'anchor_sizes': [[1.76, 0.6, 1.73]],
                'anchor_rotations': [0, 1.57],
                'anchor_bottom_heights': [-0.6],
                'align_center': False,
                'feature_map_stride': 8,
                'matched_threshold': 0.5,
                'unmatched_threshold': 0.35
            }
            - {
                'class_name': 'Motorcycle',
                'anchor_sizes': [[2.11, 0.77, 1.47]],
                'anchor_rotations': [0, 1.57],
                'anchor_bottom_heights': [-0.6],
                'align_center': False,
                'feature_map_stride': 8,
                'matched_threshold': 0.5,
                'unmatched_threshold': 0.35
            }
            - {
                'class_name': 'Bus',
                'anchor_sizes': [[11.11, 2.94, 3.47]],
                'anchor_rotations': [0, 1.57],
                'anchor_bottom_heights': [-1.78],
                'align_center': False,
                'feature_map_stride': 8,
                'matched_threshold': 0.6,
                'unmatched_threshold': 0.45
            }
            - {
                'class_name': 'Truck',
                'anchor_sizes': [[10.0, 2.6, 3.2]],
                'anchor_rotations': [0, 1.57],
                'anchor_bottom_heights': [-1.78],
                'align_center': False,
                'feature_map_stride': 8,
                'matched_threshold': 0.6,
                'unmatched_threshold': 0.45
            }
"""

# 3. Write the file
file_path = '/content/OpenPCDet/tools/cfgs/kitti_models/custom_second_original_6class.yaml'
with open(file_path, 'w') as f:
    f.write(yaml_content)

print(f"Success! The 6-class config file has been created at: {file_path}")

Success! The 6-class config file has been created at: /content/OpenPCDet/tools/cfgs/kitti_models/custom_second_original_6class.yaml


In [ ]:
# 1. Wipe any broken or empty folders
!rm -rf /content/OpenPCDet

# 2. Download a fresh copy of OpenPCDet
%cd /content
!git clone https://github.com/open-mmlab/OpenPCDet.git

# 3. Install it
%cd OpenPCDet
!pip install -r requirements.txt
!python setup.py develop

# 4. Re-link your dataset from Google Drive
!mkdir -p /content/OpenPCDet/data
!ln -s "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti" /content/OpenPCDet/data/kitti

# 5. Verify the training script exists!
!ls -l /content/OpenPCDet/tools/train.py


/content
Cloning into 'OpenPCDet'...
remote: Enumerating objects: 4175, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4175 (delta 0), reused 0 (delta 0), pack-reused 4171 (from 2)
Receiving objects: 100% (4175/4175), 4.21 MiB | 12.79 MiB/s, done.
Resolving deltas: 100% (2452/2452), done.
/content/OpenPCDet
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.7 MB/s eta 0:00:00
  Created wheel for SharedArray: filename=sharedarray-3.2.4-cp312-cp312-linux_x86_64.whl size=56617 sha256=86fb0819df6b3614d2d6fe10e939070d43af426a7e70cb9627876bc40bb8f156
  Stored in directory: /root/.cache/pip/wheels/55/52/3f/57b0bc5c3e736821bd8191dfab0919c95d57e508dd7ce3bff0
Successfully built SharedArray
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDep

In [ ]:
# 3. Create Data Links (Update these paths to match your exact Drive location)
!mkdir -p data/kitti
DRIVE_PATH = "/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original"

!ln -s "{DRIVE_PATH}/training" data/kitti/training
!ln -s "{DRIVE_PATH}/ImageSets" data/kitti/ImageSets

# 4. Redirect Output to Drive (So you don't lose checkpoints)
# This creates a folder on your drive where the model weights (.pth) will be saved
!mkdir -p "{DRIVE_PATH}/output"
!ln -s "{DRIVE_PATH}/output" /content/OpenPCDet/output

In [ ]:
!python -m pcdet.datasets.kitti.kitti_dataset create_kitti_infos \
    tools/cfgs/dataset_configs/kitti_dataset.yaml

Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 112, in _get_module_details
  File "/content/OpenPCDet/pcdet/datasets/__init__.py", line 8, in <module>
    from .dataset import DatasetTemplate
  File "/content/OpenPCDet/pcdet/datasets/dataset.py", line 10, in <module>
    from .processor.data_processor import DataProcessor
  File "/content/OpenPCDet/pcdet/datasets/processor/data_processor.py", line 6, in <module>
    import torchvision
  File "/usr/local/lib/python3.12/dist-packages/torchvision/__init__.py", line 10, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/models/__init__.py", line 2, in <module>
    from .convnext import *
  File "/usr/local/lib/python3.12/dist-packages/torchvision

In [ ]:
!pip install av2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 29.5 MB/s eta 0:00:00


In [ ]:
# 1. Comment out the import of Argo2 in the datasets init file
!sed -i 's/from .argo2.argo2_dataset import Argo2Dataset/# from .argo2.argo2_dataset import Argo2Dataset/g' /content/OpenPCDet/pcdet/datasets/__init__.py

# 2. Comment out the dictionary entry for Argo2
!sed -i "s/'Argo2Dataset': Argo2Dataset/# 'Argo2Dataset': Argo2Dataset/g" /content/OpenPCDet/pcdet/datasets/__init__.py

print("Argo2 dependencies disabled. You can now proceed with KITTI info generation.")

Argo2 dependencies disabled. You can now proceed with KITTI info generation.


In [ ]:
%cd /content/OpenPCDet
!python -m pcdet.datasets.kitti.kitti_dataset create_kitti_infos tools/cfgs/dataset_configs/kitti_dataset.yaml

/content/OpenPCDet
<frozen runpy>:128: RuntimeWarning: 'pcdet.datasets.kitti.kitti_dataset' found in sys.modules after import of package 'pcdet.datasets.kitti', but prior to execution of 'pcdet.datasets.kitti.kitti_dataset'; this may result in unpredictable behaviour
---------------Start to generate data infos---------------
train sample_idx: 000000
train sample_idx: 000003
train sample_idx: 000007
train sample_idx: 000009
train sample_idx: 000010
train sample_idx: 000011
train sample_idx: 000012
train sample_idx: 000013
train sample_idx: 000014
train sample_idx: 000016
train sample_idx: 000018
train sample_idx: 000022
train sample_idx: 000017
train sample_idx: 000026
train sample_idx: 000029
train sample_idx: 000030
train sample_idx: 000032
train sample_idx: 000034
train sample_idx: 000036
train sample_idx: 000038
train sample_idx: 000041
train sample_idx: 000043
train sample_idx: 000044
train sample_idx: 000045
train sample_idx: 000046
train sample_idx: 000049
train sample_idx: 00005

In [ ]:
%cd /content/drive/MyDrive/OpenPCDet

# Re-install in develop mode so the paths update to the Drive location
!pip install -r requirements.txt
!python setup.py develop

/content/drive/MyDrive/OpenPCDet
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for details.
        ********************************************************************************

!!
  easy_install.initialize_options(self)
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based

In [ ]:
!ls -l data/kitti

total 4
lrw------- 1 root root    0 Apr  7 11:49 'Ashish Bharti' -> '/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti'
drwx------ 2 root root 4096 Apr  7 11:50  ImageSets
lrw------- 1 root root    0 Apr  7 11:50  training -> '/content/drive/MyDrive/MES PROJECT GROUP 6/Ashish Bharti/Dataset_Original/training'


In [ ]:
!pwd
!ls

/content/drive/MyDrive/OpenPCDet
build  docker  LICENSE	pcdet		README.md	  setup.py
data   docs    output	pcdet.egg-info	requirements.txt  tools


In [ ]:
!pip install spconv-cu118  # Even on CUDA 12, the 11.8 version usually maintains backward compatibility in Colab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.7/313.7 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.7 MB/s eta 0:00:00


In [ ]:
# 1. Uninstall the broken versions
!pip uninstall -y spconv cumm

# 2. Install the pre-built versions for CUDA 12.x
# We use 'spconv-cu120' which is pre-compiled for Colab's current GPU setup
!pip install spconv-cu120 cumm-cu120

ERROR: Could not find a version that satisfies the requirement spconv-cu120 (from versions: none)
ERROR: No matching distribution found for spconv-cu120


In [ ]:
# 1. Move to the tools directory where train.py lives
%cd /content/drive/MyDrive/OpenPCDet/tools

# 2. Run the training command
!python train.py --cfg_file cfgs/kitti_models/second.yaml \
    --batch_size 4 \
    --workers 4

/content/drive/MyDrive/OpenPCDet/tools
[1/42] [GCC][c++/pch]/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h.gch
FAILED: [code=1] /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h.gch 
g++ -MMD -MT /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h.gch -MF /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h.gch.d -I "/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include" -I "/usr/local/lib/python3.12/dist-packages/pybind11/include" -I "/usr/include/python3.12" -I "/usr/local/cuda/include" -I "/usr/local/lib/python3.12/dist-packages/include" -O3 -DTV_STATIC_VARIABLE_IMPLEMENTATION -std=c++14 -fPIC -x c++-header -DTV_ENABLE_HARDWARE_ACC -c /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h -o /usr/local/lib/python3.12/dist-packages/